# PNCP — Identificação de Subenquadramento de Engenharia

TCC MBA IA & Big Data (ICMC/USP) — Profa. Solange O. Rezende

## Metodologia em 3 etapas

1. **Triagem determinística (etapa 0)** — pré-filtro lexical + verificação
   de rito de engenharia (ART/RRT, memorial, NBR…). Casos óbvios são
   reclassificados aqui, antes do ML.
2. **ML para os ambíguos** — TF-IDF + LR/RF/SVC, opcional BERTimbau.
3. **Camadas complementares** — outliers, grafos, CNAE, aditivos.

**Como usar:** rode em ordem. Cada etapa **persiste em disco** (parquet/json).
Se o kernel cair, é só continuar da célula seguinte — nada se perde.

## Célula 1 — Setup (rode 1× por sessão)

In [ ]:
!pip install -q wordcloud tqdm imbalanced-learn nltk mlxtend psutil
!pip install -q pymupdf pdfplumber pytesseract networkx python-louvain openpyxl
!pip install -q sentence-transformers transformers torch statsmodels
!apt-get install -y -qq tesseract-ocr tesseract-ocr-por > /dev/null
print('OK — dependências instaladas')

## Célula 2 — Drive + clonar o repositório

In [ ]:
import sys, os
if not os.path.exists('/content/pncp-engineering-analytics'):
    !git clone https://github.com/caiofvserra/pncp-engineering-analytics.git /content/pncp-engineering-analytics
sys.path.insert(0, '/content/pncp-engineering-analytics')

import pncp
pncp.montar_drive('/content/drive/MyDrive/PNCP_TCC')
pncp.keep_alive()
pncp.ram.monitorar_ram('início')

## Célula 3 — Coleta (parquet por ano, libera RAM entre anos)

In [ ]:
pncp.coleta.coletar(uf='SP', anos=range(2023, 2026), tamanho=500)
pncp.ram.monitorar_ram('após coleta')

## Célula 4 — EDA + texto + TF-IDF

In [ ]:
pncp.eda.executar()
from pncp import config
caminho = config.caminho(config.SUB_COLETA, 'contratos.parquet')
pncp.texto.preprocessar(caminho)
pncp.texto.marcar_termos_dominio(caminho)
pncp.texto.construir_tfidf(caminho)
pncp.ram.monitorar_ram('após texto/TF-IDF')

## Célula 5 — Triagem inicial (pré-filtro lexical)

Identifica contratos 'geral' que são *obviamente* engenharia pelo objeto.
Sem os PDFs ainda, todos viram `subenquadramento_real` provisório — vão
ser reclassificados na Célula 7 quando tivermos as features de rito.

In [ ]:
pncp.triagem.executar()

## Célula 6 — PDFs (Camada 2): TR/Edital dos óbvios + ambíguos

Prioriza os contratos marcados como `obvio_engenharia` na triagem para
saber se seguiram o rito.

In [ ]:
pncp.pdfs.executar(max_contratos=300)
pncp.ram.monitorar_ram('após PDFs')

## Célula 7 — Triagem refinada (com verificação de rito)

Re-roda a triagem cruzando com features dos PDFs:
- `rotulacao_incorreta_processo_ok` = óbvio + rito seguido (≥2 sinais)
- `subenquadramento_real` = óbvio + rito **não** seguido (violação Lei 14.133)
- `ambiguo` = vai para o ML

In [ ]:
pncp.triagem.executar()

## Célula 8 — Classificação supervisionada (ML para os ambíguos)

In [ ]:
pncp.classificacao.executar(
    fazer_grid=True,
    fazer_holdout=True,
    fazer_mcnemar=True,
    fazer_bootstrap=True,
)
pncp.ram.monitorar_ram('após classificação')

## Célula 9 — Detecção de outliers (anomalias contextuais)

Aplica IsolationForest no cluster 'geral' — contratos atípicos são
candidatos adicionais a subenquadramento.

In [ ]:
pncp.outliers.executar(fazer_iforest=True, fazer_lof=False)

## Célula 10 — Técnicas avançadas (LDA, Label Propagation, Apriori, KMeans)

In [ ]:
pncp.avancado.executar(
    fazer_lda=True,
    fazer_lp=True,
    fazer_apriori=True,
    fazer_kmeans=True,
    fazer_hier=True,
)
pncp.ram.monitorar_ram('após avançado')

## Célula 11 — Embeddings BERTimbau (opcional, mais lento)

Use `tipo='sentence-bert'` para iterar rápido; `'bertimbau'` na versão final.

In [ ]:
# pncp.embeddings.executar(tipo='sentence-bert', treinar=True)
# pncp.ram.monitorar_ram('após embeddings')

## Célula 12 — Aditivos + Grafos + CNAE

In [ ]:
pncp.aditivos.executar(max_contratos=200)
pncp.grafos.executar()
pncp.cnae.executar(
    max_consultas=200,
    caminho_excel_crea='/content/drive/MyDrive/PNCP_TCC/cnaes_crea.xlsx',
)
pncp.ram.monitorar_ram('após complementares')

## Célula 13 — Relatório final TCC + consolidação

Combina o veredito da triagem (determinístico, alta confiança) com o ranking ML.

In [ ]:
pncp.relatorio.gerar()
from pathlib import Path
rel = Path(pncp.config.PASTA_DADOS) / pncp.config.SUB_P9 / 'relatorio.md'
print(rel.read_text(encoding='utf-8'))

## Célula 14 — Validação contra ground truth manual (opcional)

1. Abra `dados/cnae/amostra_revisao_manual.csv`
2. Preencha a coluna `revisao_manual`: `subenq` / `ok` / `duv`
3. Rode esta célula

In [ ]:
csv = pncp.config.caminho(pncp.config.SUB_P8, 'amostra_revisao_manual.csv')
pncp.relatorio.validar_ground_truth(csv)

## Glossário

In [ ]:
pncp.relatorio.glossario()